# Mission 04: Logging Middleware - 실습 노트북

이 노트북은 네 번째 미션을 진행하며 에이전트 실행의 수명 주기(Lifecycle)를 추적하는 LoggingMiddleware를 설계하고 감사 로그를 파일로 적재하는 법을 실습합니다.

In [ ]:
# 1. 환경 로드
import sys
import os
from dotenv import load_dotenv

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

sys.path.append(os.path.abspath("src"))
sys.path.append(os.path.abspath("app"))
load_dotenv(override=True)

from app.utils.llm import get_llm
from app.utils.context import AgentContext
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from app.tools import web_search

### [미션 1] LoggingMiddleware 스켈레톤 구현하기

아래 빈 칸에 `before_agent`, `after_agent`, `wrap_tool_call`을 알맞게 구현하세요.

In [ ]:
import time
import json
import contextvars
from typing import Any, Dict
from langchain.agents.middleware import AgentMiddleware

class StudentLoggingMiddleware(AgentMiddleware):
    def __init__(self, log_path="./artifacts/student_audit_trail.json"):
        self.log_path = log_path
        os.makedirs(os.path.dirname(self.log_path), exist_ok=True)
        self.start_time_var = contextvars.ContextVar("agent_start_time")
        self.query_var = contextvars.ContextVar("agent_query")
        
    def before_agent(self, state: Dict[str, Any], runtime: Any) -> Dict[str, Any] | None:
        logging_enabled = getattr(runtime.context, "logging_enabled", False) if runtime and runtime.context else False
        if not logging_enabled:
            return None
            
        # TODO: 현재 시간(time.time())을 start_time_var에 저장하고,
        # 사용자의 마지막 질문 내용을 query_var에 기록한 후 콘솔에 시작 로그를 남기세요.
        return None
        
    def after_agent(self, state: Dict[str, Any], runtime: Any) -> Dict[str, Any] | None:
        logging_enabled = getattr(runtime.context, "logging_enabled", False) if runtime and runtime.context else False
        if not logging_enabled:
            return None
            
        # TODO: 소요 시간을 계산하고, 대화의 최종 답변과 히스토리를 파싱하여
        # 지정된 경로에 한 줄의 JSON(jsonl) 로그로 추가(_append_log)하세요.
        return None
        
    def wrap_tool_call(self, request, handler):
        logging_enabled = getattr(request.runtime.context, "logging_enabled", False) if request.runtime and request.runtime.context else False
        if not logging_enabled:
            return handler(request)
            
        # TODO: 도구 실행 전후의 시간을 재서 개별 도구의 latency를 감사 로그에 기록한 뒤,
        # 도구 실행 결과(handler(request))를 반환하세요.
        return handler(request)
        
    def _append_log(self, log_data):
        with open(self.log_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(log_data, ensure_ascii=False) + "\n")

### [미션 2] 미들웨어가 연동된 에이전트 생성 및 검증

작성한 미들웨어를 create_agent에 등록하고 대화를 요청하여 로그가 정확히 누적되는지 확인하세요.

In [ ]:
log_filepath = "./artifacts/student_audit_trail.json"
if os.path.exists(log_filepath):
    os.remove(log_filepath)

logging_middleware = StudentLoggingMiddleware(log_path=log_filepath)
llm = get_llm(model_name="openai:gpt-4o")

agent = create_agent(
    model=llm,
    tools=[web_search],
    middleware=[logging_middleware],
    context_schema=AgentContext
)

# 런타임 제어 컨텍스트 생성 (로깅 켜기)
context_obj = AgentContext(logging_enabled=True)

res = agent.invoke(
    {"messages": [HumanMessage(content="마케팅 기법 중 '바이럴 마케팅'에 대해 web_search 도구를 한 번 써서 간단히 검색하고 핵심 1문장으로 요약해줘.")]},
    config={"configurable": {"thread_id": "session_logging_test"}},
    context=context_obj
)
print("최종 답변:", res["messages"][-1].content)

### [미션 3] 생성된 감사 로그 검증

실제로 지정된 경로에 JSON 라인으로 로그들이 정상 저장되었는지 프린트해봅니다.

In [ ]:
if os.path.exists(log_filepath):
    print("📝 적재된 로그 내용:")
    print("-" * 80)
    with open(log_filepath, "r", encoding="utf-8") as f:
        for line in f:
            print(line.strip())
    print("-" * 80)
else:
    print("❌ 로그 파일이 생성되지 않았습니다.")